## Imports

In [1]:
from pathlib import Path
import numpy as np
import xarray as xr

from setup_lookup_table import create_lookup_table
from flood_adapt.objects.forcing.unit_system import UnitTypesLength
from flood_adapt import FloodAdapt
from flood_adapt.config.config import Settings

## Inputs 
Provide the event set and the range of SLR values to be calculated. 

In [2]:
# FloodAdapt database paths
DATA_DIR = Path(r"c:\Users\athanasi\Github\Database\Working_Databases\Charleston\4_FloodAdapt\Database")
SFINCS_BIN_PATH=Path(r"c:\Users\athanasi\Github\Database\system\win-64\sfincs_v.2.1_alpha\sfincs.exe")
FIAT_BIN_PATH=Path(r"c:\Users\athanasi\Github\Database\system\win-64\fiat_v0.2.1\fiat.exe")

# Database config
site = "charleston_beta_release"
name_event_set = "event_set_coastal"

# Scenario input parameters
slr = np.arange(0, 4, 1)
unit = UnitTypesLength.feet
fp_height = 2  # floodproof height in feet

## Initialize FloodAdapt Database

In [3]:
# Initialize FloodAdapt
settings = Settings(
    DATABASE_ROOT=DATA_DIR,
    DATABASE_NAME=site,
    SFINCS_BIN_PATH=SFINCS_BIN_PATH,
    FIAT_BIN_PATH=FIAT_BIN_PATH,
    VALIDATE_BINARIES=True,
)
fa = FloodAdapt(database_path=settings.database_path)

2025-12-15 04:11:37 PM - FloodAdapt.Database - INFO - Initializing database to charleston_beta_release at c:\users\athanasi\github\database\working_databases\charleston\4_floodadapt\database


## Create event sequence and look up table

To Do: 
- naming of projections and strategies needs to be more unique to avoid confusion when running a different analysis.
- add method to get impacts at the object ID level

In [4]:
# Lookup table
ds_impacts = create_lookup_table(
        fa, 
        name_event_set, 
        slr=slr, 
        unit = unit, 
        fp_height = fp_height, 
        )


Projection SLR_0 already exists in database
Projection SLR_1 already exists in database
Projection SLR_2 already exists in database
Projection SLR_3 already exists in database
Measure floodproof_all_0 already exists in database
Strategy no_measures already exists in database
Strategy floodproof_all_0 already exists in database
Scenario SLR_0_subevent_1_no_measures already exists in database
Scenario SLR_0_subevent_2_no_measures already exists in database
Scenario SLR_0_subevent_3_no_measures already exists in database
Scenario SLR_0_subevent_4_no_measures already exists in database
Scenario SLR_1_subevent_1_no_measures already exists in database
Scenario SLR_1_subevent_2_no_measures already exists in database
Scenario SLR_1_subevent_3_no_measures already exists in database
Scenario SLR_1_subevent_4_no_measures already exists in database
Scenario SLR_2_subevent_1_no_measures already exists in database
Scenario SLR_2_subevent_2_no_measures already exists in database
Scenario SLR_2_subeve

In [5]:
ds_impacts

<xarray.Dataset> Size: 32MB
Dimensions:       (object_id: 61858, slr: 4, strategy: 2, event: 4)
Coordinates:
  * object_id     (object_id) object 495kB '14048' '67199' ... '70438' '70444'
  * slr           (slr) int32 16B 0 1 2 3
  * strategy      (strategy) <U16 128B 'no_measures' 'floodproof_all_0'
  * event         (event) <U10 160B 'subevent_1' 'subevent_2' ... 'subevent_4'
Data variables:
    inun_depth    (object_id, slr, strategy, event) float64 16MB nan nan ... nan
    total_damage  (object_id, slr, strategy, event) float64 16MB 0.0 0.0 ... 0.0

In [6]:
ds_impacts.to_netcdf("lookup_table_charleston.nc")